In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import math

pd.set_option("display.max_columns", None)

In [ ]:
# little function to define the file root on different machines
def find_f_root(start_path: Path = Path.cwd(), anchor: str = "CASA0004_work") -> Path:
    """
    Traverse up from the start_path until the anchor folder is found. Returns the path to the anchor folder.
    """
    for parent in [start_path] + list(start_path.parents):
        if parent.name == anchor:
            return parent
    raise FileNotFoundError(f"Anchor folder '{anchor}' not found in path hierarchy.")
  
f_root = find_f_root()

# Loading the Non Victim Form file

In [ ]:
nvf_lookup_cols = [
    "C11IndivWgt", "c11indivwgt",
    "emdidc15", "emdidc19", "wmdidc14", "wmdidc19", "emdidec3", "wmdidec3","wmdidc14",
    "ecridec3", "wcridec3", "ecridc15", "wcridc14", "ecridc19", "wcridc19","wcridc14",
    "hhinc5a", "hhinc5a2", "managhh2",
    "educint",
    "nchil2",
    "polatt7", 	
    "ladsupgp",
 #    Value = 1.0	Label = Strongly agree
	# Value = 2.0	Label = Tend to agree
	# Value = 3.0	Label = Neither agree nor disagree
	# Value = 4.0	Label = Tend to disagree
	# Value = 5.0	Label = Strongly disagree
    
    "global_person_id",
    
    # mrp column candidates:
    "rnssec3",
    "rnssec5", #2024 Adult respondent Socio-Economic Classification (NS-SEC)
    #Value = 1.0	Label = Higher managerial, administrative and professional occupations
	#Value = 2.0	Label = Intermediate occupations
	#Value = 3.0	Label = Small employers and own account workers
	#Value = 4.0	Label = Lower supervisory and technical occupations
	#Value = 5.0	Label = Semi-routine and routine occupations
	#Value = 6.0	Label = Never worked and long-term unemployed
    "respsec2", #2011 Adult respondent Socio-Economic Classification (NS-SEC): Analytic Categories
	#Value = 1.1	Label = Large employer and higher managerial occupations
	#Value = 2.0	Label = Lower professional and higher technical occupations
	#Value = 3.0	Label = Intermediate occupations
	#Value = 4.0	Label = Small employers and own account workers
	#Value = 5.0	Label = Lower supervisory and technical occupations
	#Value = 6.0	Label = Semi-routine occupations
	#Value = 1.2	Label = Higher professional occupations
	#Value = 8.0	Label = Never worked
	#Value = 9.0	Label = Not classified
	#Value = 7.0	Label = Routine occupations
    
    "illharmONS2", #2024 Disability/long-standing illness ONS (2 categories)
	#Value = 1.0	Label = Not Disabled
	#Value = 2.0	Label = Disabled
    "lillharm",  #2011 Disability/long-standing illness (2 categories)
	#Value = 1.0	Label = No long standing illness
	#Value = 2.0	Label = Long standing illness
    "illharmONS", #2015 
    # Value = 1.0	Label = No long standing illness
	# Value = 2.0	Label = Long standing illness
    
    "relig3", #2024 and 2011 identical cols Adult respondent religious group

    # "rlstweek", #2024 and 2011 Respondent economic status in last week identical cols

 #    "rftpt", #2024 and 2011 Respondent working full-time or part-time identical cols 2024 REMOVED DUE TO UNSUFFICIENT SAMPLE SIZE
	# #Value = 1.0	Label = Full-time
	# #Value = 2.0	Label = Part-time
    
    "remploya", #2024 Respondent employment status
	#Value = 1.0	Label = Employed
	#Value = 2.0	Label = Unemployed
	#Value = 3.0	Label = Economically inactive
    "remploy",#2011 Respondent employment status
	#Value = 1.0	Label = Employed
	#Value = 2.0	Label = Unemployed
	#Value = 3.0	Label = Economically inactive
    
    "remploy2a", #2024 Respondent employment status (incl inactive breakdowns)
	# Value = 1.0	Label = Employed
	# Value = 2.0	Label = Unemployed
	# Value = 3.0	Label = Economically inactive: student
	# Value = 4.0	Label = Economically inactive: looking after family/home
	# Value = 5.0	Label = Economically inactive: long-term/temp sick/ill
	# Value = 6.0	Label = Economically inactive: retired
	# Value = 7.0	Label = Economically inactive: other
    "remploy2", #2011 Respondent employment status (incl inactive breakdowns)
 #    Value = 1.0	Label = Employed
	# Value = 2.0	Label = Unemployed
	# Value = 3.0	Label = Economically inactive: student
	# Value = 4.0	Label = Economically inactive: looking after family/home
	# Value = 5.0	Label = Economically inactive: long-term/temp sick/ill
	# Value = 6.0	Label = Economically inactive: retired
	# Value = 7.0	Label = Economically inactive: other
    
    "livharm1a", #2024 Respondent marital status
	# Value = -1.0	Label = Not classified
	# Value = 1.0	Label = Married/civil partnered
	# Value = 2.0	Label = Cohabiting
	# Value = 3.0	Label = Single
	# Value = 4.0	Label = Separated
	# Value = 5.0	Label = Divorced/Legally dissolved partnership
	# Value = 6.0	Label = Widowed
    "marital", #2011 Respondent marital status
	# Value = 1.0	Label = single, that is, never married
	# Value = 2.0	Label = married and living with husband/wife
	# Value = 3.0	Label = in a same-sex civil partnership and living with partner
	# Value = 4.0	Label = married and separated from husband/wife
	# Value = 5.0	Label = divorced
	# Value = 6.0	Label = widowed
    "margrp", #2011 - Respondent de facto marital status
 #    Value = 1.0	Label = Married or de facto
	# Value = 2.0	Label = Single
	# Value = 3.0	Label = Widowed
	# Value = 4.0	Label = Separated or divorced
    
    "nslivarr",#2024 Respondent ONS harmonised living arrangement	
 #    Value = -1.0	Label = Not classified
	# Value = 1.0	Label = Persons living in a couple
	# Value = 2.0	Label = Persons not living in a couple
    "livharm2", #2011 Whether respondent living in a couple
	# Value = 1.0	Label = Living in a couple
	# Value = 2.0	Label = Not living in a couple

 #    "UKborn", #2024 REMOVED DUE TO UNSUFFICIENT SAMPLE SIZE
 #    #Value = 1.0	Label = Born in the UK
	# #Value = 2.0	Label = Not born in the UK
 #    "cry2", #2024- 2011 Adult respondent country of birth
 #    #Value label information for cry2
	# #Value = 1.0	Label = England
	# #Value = 2.0	Label = Scotland
	# #Value = 3.0	Label = Wales
	# #Value = 4.0	Label = Northern Ireland
	# #Value = 5.0	Label = UK, Britain (don't know country)
	# #Value = 6.0	Label = Ireland (Republic)
	# #Value = 7.0	Label = Other
	# #Value = 8.0	Label = Refused
	# #Value = 9.0	Label = Don't know
    
    "ethgrp2a", #2024 Ethnic Group (5 categories)
    "ethgrp2", #2011 Ethnic Group (5 categories)
    
    "sex", 
    "inner", 
    # Value = 0.0	Label = Not inner city
	# Value = 1.0	Label = Inner city

    
    "rural3",#2024
    "rural2",
    #Value = 1.0	Label = Rural
	#Value = 2.0	Label = Urban

    "indlon",
    # Value = 1.0	Label = London
	# Value = 2.0	Label = Outside London
    
    "gor", "oa_sup11", "indlon", "agelong",
    "ONSpsuid", "onspsuid", "wave"

    # variables to help identify additional offences
]

In [ ]:
missing_values = ["", " ","NA", "N/A", "NaN", "nan",]
chunks = pd.read_csv(
    f_root / "data/csew/merged/Post2011_nvf.tab",
    sep="\t",
    na_values=missing_values,
    keep_default_na=True,
    usecols=lambda c: c in nvf_lookup_cols,
    dtype=str,
    chunksize=25000,
    low_memory=True
)

parts = []

for chunk in chunks:
    parts.append(chunk)

nvf_lookup = pd.concat(parts, ignore_index=True)

In [ ]:
nvf_lookup.info()

# NVF variables adjustments

## Merging columns from different years

In [ ]:
df = nvf_lookup.copy()

# "C11IndivWgt", "c11indivwgt",
# Values are identical
weight_cols = ["C11IndivWgt", "c11indivwgt"]
available = [col for col in weight_cols if col in df.columns]
if available:
    df["ind_weight"] = df[available].bfill(axis=1).iloc[:, 0]


# Ethnic group: 2011 ethgrp2 -> 2024 ethgrp2a
# Values are identical
if "ethgrp2" in df.columns:
    df["ethgrp2a"] = (
        df["ethgrp2a"].combine_first(df["ethgrp2"])
        if "ethgrp2a" in df.columns else df["ethgrp2"]
    )

#__________________________________________________________________
# IMD
# Values are identical
imd_cols = ["emdidc15", "emdidc19", "wmdidc14", "wmdidc19","emdidec3", "wmdidec3","wmdidc14"]
available = [col for col in imd_cols if col in df.columns]
if available:
    df["imd"] = df[available].bfill(axis=1).iloc[:, 0]


#__________________________________________________________________
# IMD CRIME
# Values are identical
imd_crime_cols = [ "ecridec3", "wcridec3", "ecridc15", "wcridc14", "ecridc19", "wcridc19", "wcridc14"]
available = [col for col in imd_crime_cols if col in df.columns]
if available:
    df["imd_crime"] = df[available].bfill(axis=1).iloc[:, 0]

#__________________________________________________________________
# household income, 5 categories
# Values are identical
income_cols = ["hhinc5a", "hhinc5a2"]
available = [col for col in income_cols if col in df.columns]
if available:
    df["income_group"] = df[available].bfill(axis=1).iloc[:, 0]

#__________________________________________________________________
# NS-SEC: 2011 respsec2 -> 2024 rnssec5
if "respsec2" in df.columns:
    rnssec5_2011 = df["respsec2"].map({
        "1.1": "1", "1.2": "1", "2": "1",
        "3": "2", "4": "3", "5": "4",
        "6": "5", "7": "5", "8": "6",
        "9": np.nan
    })
    df["rnssec5"] = (
        df["rnssec5"].combine_first(rnssec5_2011)
        if "rnssec5" in df.columns else rnssec5_2011
    )

#__________________________________________________________________
# Employment status: 2011 remploy -> 2024 remploya
if "remploy" in df.columns:
    df["remploya"] = (
        df["remploya"].combine_first(df["remploy"])
        if "remploya" in df.columns else df["remploy"]
    )

#__________________________________________________________________
# Detailed employment status:
# 2011 remploy2 -> 2024 remploy2a
if "remploy2" in df.columns:
    df["remploy2a"] = (
        df["remploy2a"].combine_first(df["remploy2"])
        if "remploy2a" in df.columns else df["remploy2"]
    )

#__________________________________________________________________
# Living arrangement: 2011 livharm2 -> 2024 nslivarr
if "livharm2" in df.columns:
    df["nslivarr"] = (
        df["nslivarr"].replace(-1, np.nan).combine_first(df["livharm2"])
        if "nslivarr" in df.columns else df["livharm2"]
    )

# Treat -1 in the 2024 variable as missing
if "livharm1a" in df.columns:
    df["livharm1a"] = df["livharm1a"].replace("-1", np.nan)

# First fallback: marital
if "marital" in df.columns:
    livharm1a_2011 = df["marital"].map({
        "1": "3",  # Single
        "2": "1",  # Married
        "3": "1",  # Civil partnership
        "4": "4",  # Separated
        "5": "5",  # Divorced
        "6": "6",  # Widowed
    })

    # A legally single respondent living in a couple is treated as cohabiting
    if "livharm2" in df.columns:
        livharm1a_2011.loc[
            df["marital"].eq("1") & df["livharm2"].eq("1")
        ] = "2"

    df["livharm1a"] = (
        df["livharm1a"].combine_first(livharm1a_2011)
        if "livharm1a" in df.columns else livharm1a_2011
    )

# Second fallback: margrp
if "margrp" in df.columns:
    livharm1a_from_margrp = df["margrp"].map({
        "1": "1",  # Married or de facto
        "2": "3",  # Single
        "3": "6",  # Widowed
        "4": "4",  # Separated or divorced
    })

    # margrp cannot distinguish married from cohabiting.
    # Where livharm2 says the respondent is living in a couple
    # and marital does not identify marriage/civil partnership,
    # classify as cohabiting.
    if "livharm2" in df.columns:
        not_married = (
            ~df["marital"].isin(["2", "3"])
            if "marital" in df.columns
            else pd.Series(True, index=df.index)
        )
        livharm1a_from_margrp.loc[
            df["margrp"].eq("1")
            & df["livharm2"].eq("1")
            & not_married
        ] = "2"

    df["livharm1a"] = (
        df["livharm1a"].combine_first(livharm1a_from_margrp)
        if "livharm1a" in df.columns else livharm1a_from_margrp
    )

#__________________________________________________________________
# rural - urban
if "rural2" in df.columns:
    rural2_2011 = df["rural2"].map({"1": "2", "2": "1"})
    df["rural3"] = (
        df["rural3"].combine_first(rural2_2011)
        if "rural3" in df.columns else rural2_2011
    )

#__________________________________________________________________
# Disability / long-standing illness:
if "illharmONS2" in df.columns:
    df["illharmONS2"] = pd.to_numeric(
        df["illharmONS2"], errors="coerce"
    )
else:
    df["illharmONS2"] = np.nan

for col in ["lillharm", "illharmONS"]:
    if col in df.columns:
        df["illharmONS2"] = df["illharmONS2"].combine_first(
            pd.to_numeric(df[col], errors="coerce")
        )

df["illharmONS2"] = df["illharmONS2"].astype("Int64")

#__________________________________________________________________
old_cols = [
    "respsec2", "lillharm", "illharmONS", "remploy", "remploy2",
    "marital", "margrp", "livharm2", "rural2", "ethgrp2", "cry2", 
    "emdidc15", "emdidc19", "wmdidc14", "wmdidc19","emdidec3", "wmdidec3",
    "ecridec3", "wcridec3", "ecridc15", "ecridc19", "wcridc19","wcridc14", 
    "hhinc5a", "hhinc5a2", "C11IndivWgt", "c11indivwgt"
]
df.drop(
    columns=[col for col in old_cols if col in df.columns],
    inplace=True
)

In [ ]:
df["imd_crime"].value_counts()

In [ ]:
label_maps = {
    "income_group": {
        "-1": np.nan,
        "1": "Lowest incomes",
        "2": "Lowest incomes",
        "3": "Middle incomes",
        "4": "Middle incomes",
        "5": "High incomes",
    },
    "managhh2": {
        "9": np.nan, "8": np.nan,
        "1": "Unexpected expenses -  impossible to find",
        "2": "Unexpected expenses - problem to find",
        "3": "Unexpected expenses - no problem",
    },

    "imd": {
        "1": "Most deprived decile",
        "2": "Middle deprivation deciles",
        "3": "Middle deprivation deciles",
        "4": "Middle deprivation deciles",
        "5": "Middle deprivation deciles",
        "6": "Middle deprivation deciles",
        "7": "Middle deprivation deciles",
        "8": "Middle deprivation deciles",
        "9": "Middle deprivation deciles",
        "10": "Least deprived decile"
    },
    "imd_crime": {
        "1": "Most deprived decile",
        "2": "Middle deprivation deciles",
        "3": "Middle deprivation deciles",
        "4": "Middle deprivation deciles",
        "5": "Middle deprivation deciles",
        "6": "Middle deprivation deciles",
        "7": "Middle deprivation deciles",
        "8": "Middle deprivation deciles",
        "9": "Middle deprivation deciles",
        "10": "Least deprived decile"
    },
    "educint": {
        "1": "has_qualifications",
        "2": "no_qualifications",
        "8": np.nan, "9": np.nan 
    },
    "nchil2": {
        "0": "No children in household",
        "1": "Children in household",
        "8": np.nan, "9": np.nan 
    },
    
    "rnssec3": {
        "1": "AB",
        "2": "C1",
        "3": "C2_DE",
        "4": "C2_DE",
    },
    
    # "rnssec5": {
    #     "1": "Higher managerial, administrative and professional occupations",
    #     "2": "Intermediate, routine and routine occupations",
    #     "3": "Intermediate, routine and routine occupations",
    #     "4": "Intermediate, routine and routine occupations",
    #     "5": "Intermediate, routine and routine occupations",
    #     "6": "Never worked or unemployed",
    # },
    # polatt7	Variable label = Taking everything into account I have confidence in the police in this area
    "polatt7": {
        "1": "Confident in police",
        "2": "Confident in police",
        "3": "Confident in police",
        "4": "Not confident in police",
        "5": "Not confident in police", 
        "8": np.nan, "9": np.nan 
    },

    "illharmONS2": {
        "1": "not_disabled",
        "2": "disabled",
        "98": np.nan, "99": np.nan,
    },

    "relig3": {
        "1": "no_religion",
        "2": "christian",
        "3": "non_christian_religion",
        "4": "non_christian_religion",
        "5": "non_christian_religion",
        "6": "non_christian_religion",
        "7": "non_christian_religion",
        "8": "non_christian_religion",
        "98": np.nan,
        "99": np.nan
    },

    "remploya": {
        "1": "employed",
        "2": "unemployed_or_economically_inactive",
        "3": "unemployed_or_economically_inactive",
    },

    "remploy2a": {
        "1": "Employed",
        "2": "Unemployed",
        "3": "Economically inactive", #Was: "Economically inactive: student",
        "4": "Economically inactive", #Was: "Economically inactive: looking after family/home",
        "5": "Economically inactive", #Was: "Economically inactive: long-term/temp sick/ill",
        "6": "Economically inactive", #Was: "Economically inactive: retired",
        "7": "Economically inactive", #Was: "Economically inactive: other",
    },

    "livharm1a": {
        "-1": np.nan,
        "1": "Married or cohabiting",
        "2": "Married or cohabiting",
        "3": "Single, divorced or widowed",
        "4": "Single, divorced or widowed",
        "5": "Single, divorced or widowed",
        "6": "Single, divorced or widowed",
    },

    "nslivarr": {
        "-1": np.nan,
        "1": "live_as_couple",
        "2": "not_live_as_couple",
    },

    "ethgrp2a": {
        "1": "White",
        "2": "Not white",#Was: "Mixed/multiple ethnic groups",
        "3": "Not white",#Was: "Asian/Asian British",
        "4": "Not white",#Was: "Black/African/Caribbean/Black British",
        "5": "Not white" #Was: "Other ethnic group",
    },

    "inner": {
        "0": "Not inner city",
        "1": "Inner city",
    },

    "rural3": {
        "1": "Rural",
        "2": "Urban",
    },

    "indlon": {
        "1": "London",
        "2": "Outside London",
    },

    "gor": {
        "1": "North East",
        "2": "North West",
        "3": "Yorkshire and The Humber",
        "4": "East Midlands",
        "5": "West Midlands",
        "6": "East of England",
        "7": "London",
        "8": "South East",
        "9": "South West",
        "10": "Wales",
        "11": "Scotland"
    },

    "agelong": {
        "1": "age_16_24",
        "2": "age_16_24",
        "3": "age_25_34",
        "4": "age_35_64",
        "5": "age_35_64",
        "6": "age_35_64",
        "7": "age_65_plus",
        "8": "age_65_plus",
        "9": "age_65_plus"
    },

    "oa_sup11": {
        "1": "Rural residents",
        "2": "Cosmopolitans",
        "3": "Ethnicity central",
        "4": "Multicultural metropolitans",
        "5": "Urbanites",
        "6": "Suburbanites",
        "7": "Constrained city dwellers",
        "8": "Hard-pressed living",
    },
}

for col, mapping in label_maps.items():
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .map(mapping)
            .astype("category")
        )

In [ ]:
df = df[df["gor"] != "Scotland"].copy()

## PSU codes

In [ ]:
df["onspsuid_merged"] = (
    df["ONSpsuid"]
        .combine_first(df["ONSpsuid"])
        .combine_first(df["onspsuid"]))

df = df.drop(columns=["ONSpsuid", "onspsuid"])

In [ ]:
col = "onspsuid_merged"
prefix = "2026_st_"

# Work with strings so existing and generated IDs are compared consistently
existing_ids = set(
    df[col]
    .dropna()
    .astype(str)
)

missing_mask = df[col].isna()
missing_indices = df.index[missing_mask]

new_ids = []
counter = 1

for _ in missing_indices:
    candidate = f"{prefix}{counter}"

    # Keep incrementing until the candidate is not already present
    while candidate in existing_ids:
        counter += 1
        candidate = f"{prefix}{counter}"

    new_ids.append(candidate)
    existing_ids.add(candidate)
    counter += 1

df.loc[missing_indices, col] = new_ids

In [ ]:
df.info()

## Coverage checks

In [ ]:
# Convert to numeric where possible
df["wave"] = pd.to_numeric(df["wave"],errors="coerce").astype("Int64")
wave_order = sorted(df["wave"].dropna().unique())

# Plot percentage missing by wave for every column
for col in df.columns:

    if col == "wave":
        continue

    missing_by_wave = (
        df.groupby("wave")[col]
        .apply(lambda x: x.isna().mean() * 100)
        .reindex(wave_order)
    )

    plt.figure(figsize=(8, 2))
    missing_by_wave.plot(kind="bar")

    plt.title(f"Missing values in {col} by wave")
    plt.xlabel("Wave")
    plt.ylabel("Missing values (%)")
    plt.xticks(rotation=45)
    plt.ylim(0, 100)
    plt.tight_layout()
    plt.show()

In [ ]:
for col in label_maps:
    if col in df.columns:
        print("\n", col)
        print(df[col].value_counts(dropna=False))

# Export

In [ ]:
df.to_csv(
    f_root / "data/csew/merged/nvf_lookup.csv"
)